## Preliminary Dataset Cleaning: Herculean

### 1. Dataset Import and Overview
URL: https://huggingface.co/datasets/TheFinAI/Herculean#loading

In [14]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "prices.parquet").exists():
    ROOT = ROOT / "Evelyn_Herculean"
DATA = ROOT / "data"
if not (DATA / "prices.parquet").exists():
    raise FileNotFoundError("First run download_herculean.py in the herculean directory, then open the Notebook from the project root or the herculean directory.")

prices = pd.read_parquet(DATA / "prices.parquet")
news = pd.read_parquet(DATA / "news.parquet")
filings = pd.read_parquet(DATA / "filings.parquet")

In [15]:
# dataset Overview
pd.DataFrame({
    "table": ["prices", "news", "filings"],
    "rows": [len(prices), len(news), len(filings)],
    "columns": [len(prices.columns), len(news.columns), len(filings.columns)],
})

,table,rows,columns
0,prices,2656,9
1,news,3888,4
2,filings,73,6


In [47]:

display(prices.head(7))

display(news.head(7))

display(filings.head(7))

,id,symbol,date,open,high,low,close,adj_close,volume
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000
2,3,AAPL,2024-12-04,242.869995,244.110001,241.250000,243.009995,241.689728,44383900
3,4,AAPL,2024-12-05,243.990005,244.539993,242.130005,243.039993,241.719559,40033900
4,5,AAPL,2024-12-06,242.910004,244.630005,242.080002,242.839996,241.520630,36870600
5,6,AAPL,2024-12-09,241.830002,247.240005,241.750000,246.750000,245.409409,44649200
6,7,AAPL,2024-12-10,246.889999,248.210007,245.339996,247.770004,246.423859,36914800


,id,symbol,date,highlights
0,1,AAPL,2024-12-01,"Apple’s December 1, 2024 coverage centers on i..."
1,2,AAPL,2024-12-02,"The December 2, 2024 Apple coverage portrays a..."
2,3,AAPL,2024-12-03,"demonstrated strong financial performance, wit..."
3,4,AAPL,2024-12-04,Across the December 3 coverage compiled for Ap...
4,5,AAPL,2024-12-05,- The Shacknews report highlights Apple’s firs...
5,6,AAPL,2024-12-06,"Apple Inc. is portrayed in the December 6, 202..."
6,7,AAPL,2024-12-07,"Three articles published around December 6–7, ..."


,id,symbol,date,document_type,mda_content,risk_content
0,1,AAPL,2024-02-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
1,2,AAPL,2024-05-02,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
2,3,AAPL,2024-08-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
3,4,AAPL,2024-11-01,10-K,Item 7. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
4,5,AAPL,2025-01-31,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
5,6,AAPL,2025-05-02,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...
6,7,AAPL,2025-08-01,10-Q,Item 2. Management&#8217;s Discussion and Ana...,Item 1A. Risk Factors \n\nThe Company&#8217;s...


### 4. Inspect columns and types



#### Original-data baseline


In [56]:
# Create a summary without modifying prices, news, or filings
table_map = {
    "prices": prices,
    "news": news,
    "filings": filings,
}

overview_rows = []

for table_name, frame in table_map.items():
    dates = pd.to_datetime(frame["date"], errors="raise")

    overview_rows.append({
        "Table": table_name,
        "Rows": len(frame),
        "Columns": len(frame.columns),
        "Observed date coverage": (
            f"{dates.min().date()}–{dates.max().date()}"
        ),
    })

overview = pd.DataFrame(overview_rows)
overview

,Table,Rows,Columns,Observed date coverage
0,prices,2656,9,2024-12-02–2026-03-31
1,news,3888,4,2024-12-01–2026-03-31
2,filings,73,6,2024-01-16–2026-03-25


In [57]:
# Company-level summary
pd.DataFrame({
    "prices": prices.groupby("symbol").size(),
    "news": news.groupby("symbol").size(),
    "filings": filings.groupby("symbol").size(),
}).fillna(0).astype(int)

,prices,news,filings
symbol,,,
AAPL,332,486,9
ADBE,332,486,10
AMZN,332,486,9
GOOGL,332,486,9
META,332,486,9
MSFT,332,486,9
NVDA,332,486,9
TSLA,332,486,9



There are **0 exact duplicate rows, 0 repeated IDs, and 0 repeated company/date keys** in each source table, counting occurrences beyond the first. Filings also have 0 repeated `symbol` + `date` + `document_type` keys. These are baseline checks, not reasons to remove records or guarantees about future snapshots.

Each table contains the same eight labels: `AAPL`, `ADBE`, `AMZN`, `GOOGL`, `META`, `MSFT`, `NVDA`, and `TSLA`. Prices have 332 rows per company; news has 486; filings have 10 for ADBE and 9 for each other company. Symbols have no outer whitespace. Filing categories are `10-K` (22) and `10-Q` (51), also with no outer whitespace. 



#### Column inventory and intended handling

“Null” means a value detected as missing by pandas. “Blank” means an empty or whitespace-only string; checking this condition did not trim or replace stored values. Zero nulls/blanks does not establish narrative completeness. Every nonmissing label/narrative value is a Python string; every nonmissing date value is a Python `datetime.date`.

Parquet `string` loads as pandas `str` in this inspection environment. The `object` dtype on dates contains actual dates, not date strings or mixed invalid values. No incorrect storage types were found. 



**Prices**

| Column | Parquet → loaded pandas type | Null / blank | Observed values or range | Intended handling |
| --- | --- | ---: | --- | --- |
| `id` | `int32` → `int32` | 0 / 0 | 2,656 distinct IDs | Keep `int32` as a table-local identifier. |
| `symbol` | `string` → `str` | 0 / 0 | Eight labels listed above | Keep string labels unchanged;  |
| `date` | `date32[day]` → `object` | 0 / 0 | 332 distinct dates | Keep date-only source value. |
| `open` | `double` → `float64` | 0 / 0 | Approx. 87.459999–791.150024| Keep `float64`, original precision and unit. |
| `high` | `double` → `float64` | 0 / 0 | Approx. 99.440002–796.250000| Keep `float64`, original precision and unit. |
| `low` | `double` → `float64` | 0 / 0 | Approx. 86.620003–780.820007 | Keep `float64`, original precision and unit. |
| `close` | `double` → `float64` | 0 / 0 | Approx. 94.309998–790.000000| Keep `float64`, original precision and unit. |
| `adj_close` | `double` → `float64` | 0 / 0 | Approx. 94.287781–788.148987 | Keep `float64`, original precision and unit. |
| `volume` | `int64` → `int64` | 0 / 0 | 1,166,500–818,830,900 | Keep `int64`, original unit and values. |

**News**

| Column | Parquet → loaded pandas type | Null / blank | Observed values or range | Intended handling |
| --- | --- | ---: | --- | --- |
| `id` | `int32` → `int32` | 0 / 0 | 1–3,888; 3,888 distinct IDs | Keep `int32` as a table-local identifier; no renumbering or arithmetic interpretation. |
| `symbol` | `string` → `str` | 0 / 0 | Eight labels listed above; example `AAPL` | Keep string labels unchanged; no category mapping. |
| `date` | `date32[day]` → `object` | 0 / 0 | 2024-12-01–2026-03-31; 486 distinct dates | Keep date-only source values; optional separate datetime Series, verified below. |
| `highlights` | `string` → `str` | 0 / 0 | 25–12,033 characters; 0 values with outer whitespace | Keep strings; review formatting in checklist item 5, without changing text here. |

**Filings**

| Column | Parquet → loaded pandas type | Null / blank | Observed values or range | Intended handling |
| --- | --- | ---: | --- | --- |
| `id` | `int32` → `int32` | 0 / 0 | 1–73; 73 distinct IDs | Keep `int32` as a table-local identifier; no renumbering or arithmetic interpretation. |
| `symbol` | `string` → `str` | 0 / 0 | Eight labels listed above; example `AAPL` | Keep string labels unchanged; no category mapping. |
| `date` | `date32[day]` → `object` | 0 / 0 | 2024-01-16–2026-03-25; 57 distinct dates | Keep date-only source values; optional separate datetime Series, verified below. |
| `document_type` | `string` → `str` | 0 / 0 | `10-K`: 22; `10-Q`: 51 | Keep both string categories unchanged. |
| `mda_content` | `string` → `str` | 0 / 0 | 15,930–76,907 characters; 73 values with outer whitespace | Keep strings; review formatting in checklist item 5, without changing text here. |
| `risk_content` | `string` → `str` | 0 / 0 | 333–196,794 characters; 73 values with outer whitespace | Keep strings; review formatting in checklist item 5, without changing text here. |

Numeric ranges above are rounded **only for display**; no stored value was rounded. All five price columns are finite and positive, with 0 daily range violations for open/high/low/close. Volume is already integer-valued, finite, and positive. Its source total is **159,456,800,900**, computed over all 2,656 price records. These checks do not verify the source's currency, adjustment convention, or economic accuracy; no unit change is justified by this inspection.

**Observations flagged for later review, before any action:** All 73 `mda_content` values and all 73 `risk_content` values have outer whitespace (146 cells across 73 records). These remain unchanged for the text-cleaning stage. Also, 199 adjusted-close values are outside the unadjusted daily low/high range; this comparison is recorded as an interpretation caution, not an invalid-price classification. The plan explicitly says not to force adjusted close inside that range. No suspicious value was corrected, excluded, or replaced here; a complete checklist-8 review remains separate.



#### Date-conversion rule and measured trial results

**Rule TYPE-01:** Preserve date-only source values. If later date operations need pandas datetimes, create a separate working Series and use strict conversion, preserving calendar dates and adding no timezone interpretation. Do not overwrite originals, shift dates, or interpret midnight as a known publication time. Flag any failure with its table, source ID, and original value before deciding what to do; never silently turn failures into missing values.

**Check performed:** Strict `pd.to_datetime` conversion with `errors="raise"` was tested on separate temporary Series. All three conversions succeeded; each result was checked for newly missing values and compared, after returning to calendar dates, with every original value. No source DataFrame was reassigned. The observed trial dtype was `datetime64[s]`, with no timezone; other pandas versions may choose a different resolution.

| Table | Values tested | Source nulls | Failed conversions | New missing values | Calendar-date mismatches | Trial representation |
| --- | ---: | ---: | ---: | ---: | ---: | --- |
| `prices` | 2,656 | 0 | 0 | 0 | 0 | `datetime64[s]` |
| `news` | 3,888 | 0 | 0 | 0 | 0 | `datetime64[s]` |
| `filings` | 73 | 0 | 0 | 0 | 0 | `datetime64[s]` |

No failed-conversion examples exist for this trial. Numeric, identifier, and text conversions were **not attempted**, because their existing types are suitable; their conversion-failure counts are **Not applicable**, not claimed successful conversions.

The next existing notebook cell assigns converted dates back into `prices`, `news`, and `filings`. It is unchanged and has no saved execution count. Before using it, retain original values and adapt your own working-copy approach to follow the plan. The successful temporary checks above do not mean that the notebook has been rerun or its cleaning pipeline verified.



#### Type-stage decision log

| Rule | Observed evidence | Decision and reason | Verification / actual result |
| --- | --- | --- | --- |
| TYPE-01 | All source date values are `datetime.date`, stored as `date32[day]`. | Preserve source dates; use a separate datetime Series only if an operation needs it. No malformed source dates require repair. | 6,617 successful strict trial conversions; 0 failures, new missing values, or calendar-date changes. No conversion saved. |
| TYPE-02 | IDs are `int32`, unique and nonmissing within each table. | Preserve IDs and representation to maintain traceability; do not join unrelated tables by ID. | 0 repeated IDs or source changes; no identifier conversion attempted. |
| TYPE-03 | Price fields are `float64`; volume is `int64`; observed numeric validity checks pass. | Preserve precision, values, and source units; no numeric casting, rounding, or rescaling needed. | Numeric values and volume total unchanged; source units were not independently established. |
| TYPE-04 | Labels and narratives are strings; filing text has outer whitespace. | Keep text types and category labels. Defer formatting decisions to checklist item 5. | 0 text/category changes; 146 filing cells noted for later formatting review. |



#### Checkpoint: what changed

- **Rows:** Prices 2,656 → 2,656; news 3,888 → 3,888; filings 73 → 73. Added 0; removed 0.
- **Source/loaded-table values:** Changed 0 cells and 0 records. The date trials created separate temporary representations only; equality checks confirmed the loaded source DataFrames stayed unchanged.
- **Missingness:** Every source column has 0 nulls and 0 blanks; 0 new missing values arose in the date trials. No fills were performed.
- **Keys, types, units, categories, and coverage:** Original tables unchanged; no joins, category mappings, unit conversions, or coverage exclusions performed.
- **Totals:** Source volume total remains 159,456,800,900. No IDs or price levels were summed as business metrics.
- **Review status:** Formatting observations and the adjusted-close interpretation caution above remain recorded for the relevant later steps; no corrections or resolutions are claimed.
- **Next step:** Implement any needed working-copy conversion yourself and report its result. Text standardization, a complete suspicious-value review, and a fresh notebook run remain outstanding; item 4 does not certify those later steps.

**Source-preservation evidence:** SHA-256 values were calculated before inspection and checked again afterward; all three matched. These identify the actual original files inspected, not the existing cleaned outputs.

| Original file | Unchanged SHA-256 |
| --- | --- |
| `data/prices.parquet` | `0001b983f1a302e2f7a2d2e6a6c407f44127de77e981b004fd5cce6e998a942a` |
| `data/news.parquet` | `9156b73a5c417057104abe9eeb0d9b3ea0dceea66260e5b8ef552bd182cec8e6` |
| `data/filings.parquet` | `2d9a93926e52d4e2f5f18e13d68fbdd60e14282f94df4e1c4a3f7b81daf6d95d` |


In [46]:
for frame in (prices, news, filings):
    frame["date"] = pd.to_datetime(frame["date"])

### Step 3 — Standardize text carefully (checklist 5)

**5. Standardize text.** Inspect `symbol`, `document_type`, `highlights`, `mda_content`, and `risk_content` for whitespace, inconsistent labels, blank strings, and formatting artifacts. Count affected values before choosing rules.

**Hints:** The displayed filing excerpts contain HTML entities such as `&#8217;`. Inspect actual examples and consider whether decoding would improve readability without changing meaning. Separate label normalization from narrative-text cleaning: punctuation, capitalization, negation, numbers, and paragraph boundaries may carry meaning. Preserve identifiers and any leading zeros. Do not assume every text field needs the same treatment.

**Fill in:** [Field → observed pattern → proposed rule → reason → affected count]. Show selected original and cleaned text side by side, and explain which meaningful differences you preserved. Check whether standardization creates new duplicate keys or blank values.

**Item 5 status and evidence:** [ ]. Write and run your own code below after recording your rules.

### Step 4 — Investigate duplicates and missingness (checklist 6–7)

**6. Investigate duplicates.** Count exact duplicate rows, repeated `id` values, and repeated candidate business keys separately in each table. State whether each count measures duplicate groups, all involved rows, or extra rows beyond the first. Inspect conflicting records before deciding what to retain.

**Hints:** Similar news text or repeated filing language does not automatically mean a duplicate. Multiple records for the same company and date may be legitimate. If records conflict, what evidence supports choosing one? Make any keep/remove rule deterministic and document exceptions. Repeat key checks after text and type normalization.

**Fill in:** [Table → duplicate category → count → examples → keep/remove rule and reason → final count].

**7. Handle missing data.** Count missing values per column across the full tables, including blanks or placeholder strings you have verified actually mean “missing.” Distinguish absent source values, values made missing by conversion, and gaps introduced by a later merge.

**Hints:** Missing prices, missing text, and missing company labels have different consequences. An absent price row on a non-trading day differs from a missing value in an existing row. Avoid filling a missing numeric value with zero unless zero has the intended meaning. If considering carrying a value forward, justify the meaning, company grouping, ordering, and maximum gap. Explain the effect of dropping observations on coverage and your metric.

**Fill in:** [Column → missing before → chosen handling and reason → missing after → rows affected]. Revisit this table after merging, if applicable.

**Separate statuses and evidence for items 6 and 7:** [ ]. Add your own code and findings below.

### Step 5 — Investigate suspicious records (checklist 8)

**8. Check suspicious values.** Write plausible validity rules, then count and inspect exceptions across the full dataset.

- **Dates and coverage:** Check failed dates, minimum/maximum dates by company and table, unexpected gaps, and records outside your intended scope. Distinguish trading-day prices from calendar-day news and occasional filings. Verify what each date represents before interpreting alignment.
- **Prices and volume:** Investigate nonpositive prices, negative or non-whole volume, and inconsistent daily ranges. Should the low be no greater than the open/close and the high no less? Investigate large price changes. Do not force adjusted close inside the unadjusted daily range; the fields may use different adjustment bases.
- **Labels and text:** Review unexpected symbols or document types, empty or unusually short/long narratives, and any mismatch you notice between a text's dates or company references and its labels.

These are candidate checks, not evidence that such problems exist. An extreme observation may be real. Use supporting evidence before correcting or removing it; otherwise flag it and describe the limitation.

**Fill in:** [Check → rationale → number of exceptions → examples → corrected / kept / flagged → evidence]. Record all decisions in the log.

**Item 8 status and evidence:** [ ]. Add your own checks below.

### Step 6 — Combine tables only if your question needs it (checklist 9)

**9. Verify merges.** First decide whether a merge is necessary. If you keep tables separate, explain why and mark this item Not applicable with that explanation.

If you merge, state the intended output row, join keys, join type, and expected relationship (one-to-one, many-to-one, or another justified relationship). Test the keys before merging; do not use unrelated table IDs as cross-table keys.

**Hints:** Prices and news may have different calendar coverage. Several filings for one company can multiply rows in a naive company-only merge. Decide how to handle multiple matching records and unmatched records before joining. If assigning the most recent filing to a price date, investigate when that filing became publicly available. If studying future price changes, avoid using information that was not available at the chosen observation time. Date-only data may leave timing uncertainty you must document.

**Fill in:** [Tables → output row meaning → keys/time rule → expected relationship → reason]. Report row counts before and after, unmatched keys and affected rows on both sides, unexpected row multiplication, and uniqueness of the resulting key. Show representative matched and unmatched cases; explain intentional exclusions or expansion.

**Item 9 status and evidence:** [ ]. Add your own merge and validation code only if applicable.

### Step 7 — Reconcile counts and check your metric (checklist 10–11)

**10. Reconcile changes.** Build a before/after audit for each table and any merged result. Explain every change in row count, key count, and important totals. Count removals at the step where they happen so overlapping problems are not counted twice. A flag or corrected cell does not necessarily remove a row.

**Fill in:** [Starting rows → rows removed by each reason → rows intentionally added/expanded → final rows]. Reconcile overall results and relevant company/date groups. Choose meaningful totals, such as volume if relevant, and explain changes caused by exclusions or duplicated rows after a merge. Do not sum identifiers as a business total. Unchanged data still needs evidence.

**11. Check calculations.** Define your chosen metric in words, its units, numerator/denominator where applicable, eligible records, and treatment of missing values. Report the contributing record count for **each metric and group**, including excluded records. State explicitly how you handle missing group keys and the difference between no observations and a genuine zero result.

**Hints:** If calculating returns, explain which price series you use, sort within company, check the previous observation's date, and handle the first observation or a missing/zero denominator. If comparing groups, consider whether their time coverage and sample sizes are comparable. For all-missing groups, verify that your result has the meaning you intended. Independently check a small calculation by hand.

**Fill in:** [Metric → eligible population → missing-value rule → missing-group-key rule → contributing count → independent check].

**Separate statuses and evidence for items 10 and 11:** [ ]. Add your own audit and calculations below.

### Step 8 — Verify five records and one Codex suggestion (checklist 12)

**12. Verify individual records.** Select five traceable records, including difficult cases found during inspection. Possibilities include formatting artifacts, a conversion problem, a duplicate candidate, an unmatched date, or an unusual price; use cases that actually exist. Include a normal record that should remain unchanged if helpful. Do not invent source problems just to fill categories.

**Before inspecting the cleaned result**, write the expected outcome and reason from your rules. Then compare it with the actual result. Identify each record by table and a stable source identifier/key; for a removed record, show its absence and the removal reason.

| Source table and record key | Original value(s) | Expected result, written first | Reason | Actual result | Matches? / follow-up |
| --- | --- | --- | --- | --- | --- |
| [Record 1] | [ ] | [ ] | [ ] | [ ] | [ ] |
| [Record 2] | [ ] | [ ] | [ ] | [ ] | [ ] |
| [Record 3] | [ ] | [ ] | [ ] | [ ] | [ ] |
| [Record 4] | [ ] | [ ] | [ ] | [ ] | [ ] |
| [Record 5] | [ ] | [ ] | [ ] | [ ] | [ ] |

**Required Codex reflection:** These notebook prompts are Codex assistance. Pick one suggestion you evaluated, such as decoding an HTML entity or testing a candidate key. Describe the suggestion, your independent check against the original data, the result, and whether you accepted, changed, or rejected it. Showing that code ran is not enough to establish that the rule is appropriate.

**Fill in:** Suggestion: [ ] · Independent check: [ ] · Evidence: [ ] · Decision: [ ].

**Item 12 status and evidence:** [ ]. The five examples supplement the full-dataset checks above.

### Step 9 — Reproduce the work and explain limits (checklist 13–14)

**13. Test reproducibility.** Once your own implementation is complete, restart the kernel and run all cells from the unchanged original data in order. Check that the notebook works without variables from a previous session. Record the environment/dependencies, working directory, exact run steps, and expected output files. Explain how you compared regenerated outputs with the submitted results, including counts, schemas, and relevant values. Check that the raw files remain unchanged.

**Fill in:** Dependencies/versions: [ ] · Run steps: [ ] · Output files: [ ] · Comparison method and result: [ ] · Raw-file preservation result: [ ].

**14. Document remaining limitations.** List unresolved quality issues and limitations in coverage or interpretation. Consider uncertain date meanings, missing publication times, incomplete text, unavailable records, excluded observations, or unequal coverage only where supported by your inspection. Explain how each could affect your business answer and what additional information would resolve it. Do not label an unresolved issue “Done” merely because you documented it.

**Fill in:** [Issue → affected data → possible effect on conclusion → current handling → information still needed].

**Separate statuses and evidence for items 13 and 14:** [ ].

### Step 10 — Assemble your individual submission

This section summarizes the PDF's deliverables; it is a checklist for you to complete after doing the work.

- [ ] **Individual PDF for Gradescope:** Include all 14 numbered items with status and evidence, the cleaning decision log (rule, reason, check), five original/expected/actual record checks, and your explanation of how you checked one Codex suggestion. Submit by the posted deadline.
- [ ] **Cleaned sample:** Include up to **50 cleaned rows**, including difficult cases, and explain how you selected them. For multiple tables, label each sample row's source and keep the combined sample within 50 rows. If real data cannot be shared, provide a clearly labeled synthetic sample and explain how the instructor can review the real results through an approved channel.
- [ ] **Your own branch in the team's project repository:** Include your cleaning scripts (`.py`), notebook (`.ipynb`), sample, dependency files, and README. Each student must use a different branch; do not submit to `main` or the shared course repository. This guide does not create scripts or publish your work.
- [ ] **README:** Record source and version, run instructions, and expected output files. Make the submitted work reproducible from the originals.
- [ ] **In both the PDF and README:** Exact branch name: [ ] · Branch URL: [ ] · Submitted commit ID: [ ]. Make sure the instructor has repository access.

**Final self-check:** Can a reviewer trace each cleaning decision to a reason, a full-dataset verification result, and a reproducible output? Replace every placeholder before submission, and explain every Not applicable or Unresolved status.